#

In [1]:
from pyIMSRG import *
import numpy as np


emax =3         # maximum number of oscillator quanta in the model space
ref = 'O22'     # reference used for normal ordering
val = ref # valence space

core_generator = 'atan'   # definition of generator eta for decoupling the core (could also use 'white')
smax_core = 50       # limit of integration in flow parameter s for first stage of decoupling
#smax_core = 0       # limit of integration in flow parameter s for first stage of decoupling

##### Example format of how to read input interaction matrix elements from file (these are not included with the code)
#f2b='input/TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw16_emax14_e2max28.me2j.gz'
#f2e1,f2e2,f2l = 14,28,14
#f3b='input/NO2B_ThBME_EM1.8_2.0_3NFJmax15_IS_hw16_ms18_36_18.stream.bin'
#f3e1,f3e2,f3e3 = 18,36,18
#mode3n='no2b'
#LECs = 'EM1820'
#hw=16

#### Example format of how to read input interaction matrix elements from file (these are not included with the code)
#f2b='input/TwBME-HO_NN-only_N3LO_EM500_srg1.8_hw16_emax14_e2max28.me2j.gz'
f2b='input/chi2b_srg0625_eMax08_hwHO020.me2j.gz'
f2e1,f2e2,f2l = 4,8,4
f3b='none'
f3e1,f3e2,f3e3 = 14,28,18
mode3n='no2b'
LECs = 'EM7.5_1820'
hw=20



#hw = 20    # harmonic oscillator basis frequency
#LECs='Minnesota'
#f3b='none'

##########################################################################
###  END PARAMETER SETTING. BEGIN ACTUALLY DOING STUFF ##################
##########################################################################


### Create an instance of the ModelSpace class
ms = ModelSpace(emax,ref,val)
ms.SetHbarOmega(hw)

### the ReadWrite object handles reading and writing of files
rw = ReadWrite()

rank_j, parity, rank_Tz, particle_rank = 0,0,0,2
if f3b != 'none':
   particle_rank = 3

### Create an instance of the Operator class, representing the Hamiltonian
H = Operator(ms,rank_j, parity, rank_Tz, particle_rank)

### Either generate the matrix elements of the Minnesota potential, or read in matrix elements from file
if LECs == 'Minnesota':
    H += OperatorFromString(ms,'VMinnesota')

else:
  ### Read Two-body matrix elements
  rw.ReadBareTBME_Darmstadt(f2b,H,f2e1,f2e2,f2l)
  ### Read Three-body matrix elements
  if f3b != 'none':
     if mode3n == 'no2b':
        H.ThreeBody.SetMode('no2b')
        H.ThreeBody.ReadFile([f3b],[f3e1,f3e2,f3e3])
     else:
        rw.Read_Darmstadt_3body(f3b,H,f3e1,f3e2,f3e3)


### Add the relative kinetic energy, so H = Trel + V
H += OperatorFromString(ms,'Trel')
print('after reading files, 3-body norm is',H.ThreeBodyNorm())

### Create an instance of the HartreeFock class, used for solving the Hartree-Fock equations
hf = HartreeFock(H)
hf.Solve()
hf.PrintSPEandWF()

### Do normal ordering with respect to the HF basis, and retain only up to 2-body operators
HNO = hf.GetNormalOrderedH(2)

### Create an instance of the IMSRGSolver class, used for solving the IMSRG flow equations
imsrgsolver = IMSRGSolver(HNO)
imsrgsolver.SetMethod('magnus')  # Solve using the Magnus formulation. Could also be 'flow_RK4'

imsrgsolver.SetGenerator(core_generator)
imsrgsolver.SetSmax(smax_core)

### Do the first stage of integration to decouple the core
imsrgsolver.Solve()


### Hs is the IMSRG-evolved Hamiltonian
Hs = imsrgsolver.GetH_s()


Read 5696 matrix elements 
after reading files, 3-body norm is 0.0
Calculating moshinsky with Lmax = 3
done calculating moshinsky (389 elements)
Hash table has 397 buckets and a load factor 0.979849  estimated storage ~ 1.17123e-05 GB
HF converged after 28 iterations. 
e1hf = 586.8616468
e2hf = -760.4509370
e3hf = 0.0000000
EHF = -173.5892901
  i:   n   l  2j 2tz            SPE         occ.   |    overlaps
  0:   0   0   1  -1     -87.412541     1.000000   |  0.997979   0.063541  
  1:   0   0   1   1     -85.061803     1.000000   |  0.994698   0.102835  
  2:   0   1   3  -1     -48.260889     1.000000   |  0.996071   0.088559  
  3:   0   1   3   1     -42.809251     1.000000   |  0.994952   0.100348  
  4:   0   1   1  -1     -42.862881     1.000000   |  0.998255   0.059044  
  5:   0   1   1   1     -38.600111     1.000000   |  0.994513   0.104617  
  6:   0   2   5  -1     -12.967860     0.000000   |  1.000000  
  7:   0   2   5   1     -10.314165     1.000000   |  1.000000  
  8:

In [2]:
cm=Commutator
gm=Generator()

In [3]:
## initialize the T and D^dagger

def htc(Haml, chi):
    
    ## generate a antihermit chi
    chi_d = gm.GetEOM_ladder(chi, 1)
    
    ht_plus= chi*0
    ht_minus= chi*0
    
    ht_plus.SetAntiHermitian()
    
    ht_minus.SetHermitian()
    
    ht_plus = cm.Commutator(Haml, chi )
    ht_minus = cm.Commutator(Haml, chi_d )

    
    heom1= gm.GetEOM_ladder(ht_plus, 0)
    
    heom2= gm.GetEOM_ladder(ht_minus, 0)
    
    hod = (heom1+heom2)/2

    hod.SetHermitian()

    return(hod)




def Norm(T1, T2):
    return(gm.GetEOM_Overlap(T1,T2))

import numpy as np

def lanczos_proc( hv_func, norm_func, haml, vi, ndim):
    lanczos_vector = []
    hall = np.zeros([ndim,ndim])
    hall[0,0]=0.

    ## normalize it to 1
    nn=norm_func(vi,vi)
    print(nn)
    vi=vi/np.sqrt(nn)
    lanczos_vector.append(vi)

    for j in range(ndim):
        
        w = hv_func(haml,lanczos_vector[j])
    
        ai=norm_func(w,lanczos_vector[j])
        
    
        if(j>0):
            w=w-ai*lanczos_vector[j]-bj*lanczos_vector[j-1]
        else:
            w=w-ai*lanczos_vector[j]
        
        hall[j,j]=ai


        bj = np.sqrt(norm_func(w,w))
        #print(j,Norm(w,w), norm_func(w,w))
        #print(j, norm_func(w,w),ai,bj)
        if bj < 0.00001 :
            break
        lanczos_vector.append(w/bj)
    
        if(j<ndim-1):
            hall[j,j+1]=bj
            hall[j+1,j]=bj
        #print(j,ai,bj)
   # print(hall)
    e,v = np.linalg.eig(hall[0:j,0:j])
    
    return(e,v,lanczos_vector)


In [46]:
unt = UnitTest(ms)
rank_j, parity, rank_Tz, particle_rank, herm= 2,0,0,2,1
 
h3= unt.RandomOp( ms, rank_j,  rank_Tz, parity, particle_rank,herm)
chi= gm.GetEOM_ladder(h3,0)

#chid.PrintTwoBody_ch(20)
cnorm=Norm(chi,chi)
chi=chi/np.sqrt(cnorm)
cnorm=Norm(chi,chi)
print(cnorm)


chid= gm.GetEOM_ladder(chi,1)
cnorm=Norm(chid,chid)
print(cnorm)

0.9999999999999992In  RandomOp  norm of 1b : 42.810738095144   norm of 2b 3418.536461590664

0.9999999999999992


In [47]:
hp=chid*0
hm=chi*0

In [48]:
hp=cm.Commutator(Hs, chi)
hm=cm.Commutator(Hs,chid)

In [72]:
heomp= gm.GetEOM_ladder(hp, 0)
heomm= gm.GetEOM_ladder(hm, 0)

In [82]:
heomp.PrintTwoBody_ch(8)

0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 4.657159193752 0.000000000000 -1.280114596193 -3.889353616471 -2.461034842435 2.092256328384 6.355255077540 -3.075025518560 -1.607619885812 -6.983144029770 -6.036280511921 -2.539314568394 -9.772846464982 7.801308517254 0.933177159540 9.579062358012
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0

In [81]:
heomm.PrintTwoBody_ch(8)

0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 -0.703845126568 -2.798943785545 -4.674486229810 -3.263695313904 5.174308742309
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000 0.000000000000
0.000000000000 0.0000